# 06 — Interactive Demo
### DiscoverAI · Deloitte x LUISS

Gradio web interface integrating the full DiscoverAI pipeline.

Prerequisites: run notebooks 01-04 first.
Notebook 05 optional (summaries shown if product_profiles.csv exists).

Tabs:
- Search: free text query with price/rating filters and BART summaries
- Similar products: content-based recommendation
- System info: index stats, model info, coverage


## 0 . Setup

In [1]:
import os, sys, warnings
import numpy as np
import pandas as pd
import faiss
import gradio as gr
import torch
from sentence_transformers import SentenceTransformer

warnings.filterwarnings('ignore')

DATA_DIR = 'data'
sys.path.insert(0, 'src')

comb_emb = np.load(os.path.join(DATA_DIR, 'combined_embeddings.npy'))
index    = faiss.read_index(os.path.join(DATA_DIR, 'faiss_index.bin'))

profiles_path = os.path.join(DATA_DIR, 'product_profiles.csv')
index_path    = os.path.join(DATA_DIR, 'embedding_index_enriched.csv')
idx = pd.read_csv(profiles_path if os.path.exists(profiles_path) else index_path)

if   torch.backends.mps.is_available(): DEVICE = 'mps'
elif torch.cuda.is_available():         DEVICE = 'cuda'
else:                                   DEVICE = 'cpu'

model = SentenceTransformer('all-mpnet-base-v2', device=DEVICE)

has_summaries = 'summary_full' in idx.columns and idx['summary_full'].notna().any()
has_entities  = 'ingredients'  in idx.columns and idx['ingredients'].notna().any()

print(f'Device: {DEVICE}  |  Products: {len(idx):,}')
print(f'Summaries: {"OK" if has_summaries else "NO"}  |  Entities: {"OK" if has_entities else "NO"}')


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

MPNetModel LOAD REPORT from: sentence-transformers/all-mpnet-base-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Device: mps  |  Products: 7,604
Summaries: NO  |  Entities: NO


In [3]:
import json

IMG_PATH  = os.path.join(DATA_DIR, 'product_images.csv')
META_PATH = os.path.join(DATA_DIR, 'meta_Health_and_Personal_Care.json')

if not os.path.exists(IMG_PATH) and os.path.exists(META_PATH):
    print('Building product image map (one-time, ~30s)...')
    asins_needed = set(idx['parent_asin'])
    rows = []
    with open(META_PATH) as f:
        for line in f:
            try:
                rec = json.loads(line)
            except json.JSONDecodeError:
                continue
            asin = rec.get('parent_asin')
            if asin not in asins_needed:
                continue
            imgs = rec.get('images', []) or []
            url = None
            for img in imgs:
                if img.get('variant') == 'MAIN':
                    url = img.get('large') or img.get('hi_res') or img.get('thumb')
                    break
            if url is None and imgs:
                url = imgs[0].get('large') or imgs[0].get('hi_res') or imgs[0].get('thumb')
            if url:
                rows.append({'parent_asin': asin, 'image_url': url})
    pd.DataFrame(rows).to_csv(IMG_PATH, index=False)
    print(f'Saved {len(rows):,} image URLs to {IMG_PATH}')

if os.path.exists(IMG_PATH):
    img_df = pd.read_csv(IMG_PATH)
    if 'image_url' in idx.columns:
        idx = idx.drop(columns=['image_url'])
    idx = idx.merge(img_df, on='parent_asin', how='left')
    has_images = idx['image_url'].notna().any()
    print(f'Products with image: {idx["image_url"].notna().sum():,} / {len(idx):,}')
else:
    has_images = False
    print('No image data available (meta JSON missing).')


Building product image map (one-time, ~30s)...
Saved 7,604 image URLs to data/product_images.csv
Products with image: 7,604 / 7,604


## 1 . Quality scores

Computed here if product_profiles.csv does not already contain them.

In [4]:
if 'quality_score' not in idx.columns:
    reviews = pd.read_csv(os.path.join(DATA_DIR, 'reviews_cleaned.csv'))
    rev_agg = reviews.groupby('parent_asin').agg(
        n_reviews=('rating','count'),
        pct_positive=('rating', lambda x: (x>=4).mean()),
        pct_negative=('rating', lambda x: (x<=2).mean()),
        total_helpful=('helpful_vote','sum'),
    ).reset_index()
    idx = idx.merge(rev_agg, on='parent_asin', how='left')

    rating_norm  = (idx['product_avg_rating'].fillna(3)-1)/4
    sentiment    = idx['pct_positive'].fillna(0.5) - 0.5*idx['pct_negative'].fillna(0.5)
    hv_log       = np.log1p(idx['total_helpful'].fillna(0))
    helpful_cred = (hv_log/hv_log.quantile(0.95)).clip(upper=1.0)
    idx['quality_score']    = 0.4*rating_norm + 0.35*sentiment + 0.25*helpful_cred

    rc_log = np.log1p(idx['product_rating_count'].fillna(0))
    idx['popularity_score'] = (rc_log/rc_log.quantile(0.95)).clip(upper=1.0)

    for col in ['quality_score','popularity_score']:
        mn, mx = idx[col].min(), idx[col].max()
        idx[col] = (idx[col]-mn)/(mx-mn+1e-9)

print('Quality scores ready.')


Quality scores ready.


## 2 . Search & recommendation

All logic lives in `src/mean_squared_terrors/search.py`. Two thin wrappers (`search`, `recommend`) close over the notebook globals.

In [5]:
from mean_squared_terrors.search import (
    parse_query_v2,
    expand_query,
    search_v3 as _search_v3,
    recommend_similar as _recommend_similar,
)


def search(query, k=5, price_buckets=None, min_rating=3.5, verbose=False):
    return _search_v3(
        query, model, index, idx,
        k=k, price_buckets=price_buckets, min_rating=min_rating, verbose=verbose,
    )


def recommend(query, k=5):
    return _recommend_similar(query, model, index, idx, comb_emb, k=k)


print('Search functions loaded from src/mean_squared_terrors/search.py')


Search functions loaded from src/mean_squared_terrors/search.py


## 3 . HTML result formatters

In [2]:
TABLE_CSS = (
    '<style>'
    '.rtbl{width:100%;border-collapse:collapse;font-size:13px}'
    '.rtbl th{padding:8px 10px;text-align:left;border-bottom:2px solid #e2e8f0;'
    'color:#718096;font-weight:500;font-size:12px}'
    '.rtbl td{padding:8px 10px;border-bottom:1px solid #f7fafc;vertical-align:top}'
    '.rtbl tr:hover td{background:#f7fafc}'
    '.sc{background:#ebf8ff;color:#2b6cb0;padding:2px 7px;border-radius:8px;'
    'font-size:11px;font-weight:600}'
    '.scd{background:#f0fff4;color:#276749;padding:2px 6px;border-radius:8px;'
    'font-size:10px;font-weight:600}'
    '.sm{font-size:11px;color:#718096;margin-top:4px;line-height:1.5}'
    '.pr{color:#276749;font-size:11px}'
    '.cn{color:#9b2c2c;font-size:11px}'
    '.thumb{width:64px;height:64px;object-fit:cover;border-radius:6px;'
    'flex-shrink:0;border:1px solid #e2e8f0;background:#f7fafc}'
    '.prodcell{display:flex;gap:12px;align-items:flex-start}'
    '</style>'
)


def _img_tag(url, css_class='thumb'):
    if url is None or pd.isna(url) or not str(url).strip():
        return ''
    return (f"<img src='{url}' class='{css_class}' loading='lazy' "
            f"onerror=\"this.style.display='none'\" />")


def format_results(df, show_scores=False):
    if df is None or len(df) == 0:
        return "<p style='color:#718096;padding:1rem'>No results.</p>"

    df = df.reset_index(drop=True)

    html = TABLE_CSS + "<table class='rtbl'><tr><th>#</th><th>Product</th>"
    html += "<th>Rating</th><th>Price</th>"
    if show_scores:
        html += "<th>Sim</th><th>Qual</th><th>Pop</th>"
    html += "<th>Score</th></tr>"

    for i, row in df.iterrows():
        title  = str(row.get('product_title',''))[:65]
        brand  = str(row.get('brand','')) if pd.notna(row.get('brand')) else ''
        rating = f"&#11088; {row['product_avg_rating']:.1f}" if pd.notna(row.get('product_avg_rating')) else 'N/A'
        price  = f"${row['price']:.2f}" if pd.notna(row.get('price')) else '&#8212;'
        score  = f"{row.get('score', row.get('similarity',0)):.3f}"
        nrev   = f" <small style='color:#a0aec0'>({int(row['n_reviews'])}rev)</small>" if pd.notna(row.get('n_reviews')) else ''

        summ = ''
        if has_summaries and pd.notna(row.get('summary_full')) and str(row.get('summary_full','')).strip():
            s = str(row['summary_full'])[:150]
            dots = '...' if len(str(row['summary_full']))>150 else ''
            summ += f"<div class='sm'>{s}{dots}</div>"
            if pd.notna(row.get('pros')) and str(row.get('pros','')).strip():
                summ += f"<div class='pr'>+ {str(row['pros'])[:100]}</div>"
            if pd.notna(row.get('cons')) and str(row.get('cons','')).strip():
                summ += f"<div class='cn'>- {str(row['cons'])[:100]}</div>"

        brd = f"<br><small style='color:#a0aec0'>{brand}</small>" if brand else ''
        img_html = _img_tag(row.get('image_url'))
        text_block = f"<div><b>{title}</b>{brd}{nrev}{summ}</div>"
        prod_cell  = f"<div class='prodcell'>{img_html}{text_block}</div>"

        html += f"<tr><td><b>{i+1}</b></td><td>{prod_cell}</td>"
        html += f"<td>{rating}</td><td style='color:#276749'>{price}</td>"

        if show_scores:
            sim  = f"{row.get('similarity', 0):.3f}"      if pd.notna(row.get('similarity'))       else '&#8212;'
            qual = f"{row.get('quality_score', 0):.3f}"   if pd.notna(row.get('quality_score'))    else '&#8212;'
            pop  = f"{row.get('popularity_score', 0):.3f}" if pd.notna(row.get('popularity_score')) else '&#8212;'
            html += f"<td><span class='scd'>{sim}</span></td>"
            html += f"<td><span class='scd'>{qual}</span></td>"
            html += f"<td><span class='scd'>{pop}</span></td>"

        html += f"<td><span class='sc'>{score}</span></td></tr>"
    html += '</table>'
    return html


def format_detail(asin):
    row = idx[idx['parent_asin'] == asin]
    if len(row) == 0:
        return ''
    row = row.iloc[0]

    img_html = ''
    if pd.notna(row.get('image_url')):
        img_html = (f"<div style='float:right;margin:0 0 12px 16px'>"
                    f"<img src='{row['image_url']}' loading='lazy' "
                    f"style='width:160px;height:160px;object-fit:cover;border-radius:8px;"
                    f"border:1px solid #e2e8f0' onerror=\"this.style.display='none'\" />"
                    f"</div>")

    h  = f"<div style='padding:8px 0;overflow:auto'>{img_html}"
    h += f"<h3 style='margin:0 0 4px;font-size:15px'>{str(row['product_title'])[:100]}</h3>"
    if pd.notna(row.get('brand')):
        h += f"<p style='color:#718096;font-size:13px;margin:0'>Brand: {row['brand']}</p>"
    if pd.notna(row.get('price')):
        h += f"<p style='font-size:13px;margin:4px 0'>&#128176; ${row['price']:.2f} ({row.get('price_bucket','')})</p>"

    nrev = int(row['n_reviews']) if pd.notna(row.get('n_reviews')) else '?'
    h += f"<p style='font-size:13px;margin:4px 0'>&#11088; {row.get('product_avg_rating','?')} &middot; {nrev} reviews</p>"

    if has_summaries and pd.notna(row.get('summary_full')):
        h += "<hr style='border:0.5px solid #e2e8f0;margin:8px 0'>"
        h += f"<p style='font-size:13px'><b>Summary:</b> {row['summary_full']}</p>"
        if pd.notna(row.get('pros')) and row['pros']:
            h += f"<p style='font-size:12px;color:#276749'>+ {row['pros']}</p>"
        if pd.notna(row.get('cons')) and row['cons']:
            h += f"<p style='font-size:12px;color:#9b2c2c'>- {row['cons']}</p>"

    if has_entities and pd.notna(row.get('ingredients')) and row['ingredients']:
        ings = str(row['ingredients']).replace('|', ' &middot; ')[:200]
        h += f"<p style='font-size:12px'><b>Ingredients:</b> {ings}</p>"

    h += '</div>'
    return h


print('Formatters ready.')


Formatters ready.


## 4 . Gradio UI layout

Three tabs: Search, Similar products, System info.

In [8]:
import traceback

ERROR_HTML = (
    "<div style='padding:12px;border:1px solid #fed7d7;background:#fff5f5;"
    "border-radius:6px;font-size:13px;color:#9b2c2c'>"
    "<b>Search error.</b><br>"
    "<code style='font-size:11px;color:#742a2a'>{}</code></div>"
)


def do_search(query, n, price, rating_s, show_sc):
    if not query.strip():
        return '<p>Enter a query.</p>', ''
    try:
        parsed   = parse_query_v2(query)
        expanded = expand_query(query)
        buckets  = [price.lower()] if price != 'All' else parsed['price_buckets']
        min_r    = float(rating_s) if rating_s != 'None' else None
        results  = search(query, k=int(n), price_buckets=buckets, min_rating=min_r)

        if len(results) == 0:
            return ("<p style='color:#718096'>No results. "
                    "Try lowering the minimum rating.</p>", '')

        info  = ("<div style='font-size:12px;color:#718096;margin-bottom:8px;"
                 "padding:6px 10px;background:#f7fafc;border-radius:6px'>")
        if expanded.lower().strip() != query.lower().strip():
            info += f'Expansion: <i>{query}</i> &#8594; <i>{expanded}</i>&nbsp;&nbsp;'
        if buckets:                  info += f'Price: {buckets}&nbsp;&nbsp;'
        if parsed['quality_boost']:  info += 'Quality boost&nbsp;&nbsp;'
        if parsed['exclude_words']:  info += f"Exclude: {parsed['exclude_words']}"
        info += '</div>'

        detail = format_detail(results.iloc[0]['parent_asin'])
        return info + format_results(results, show_scores=bool(show_sc)), detail
    except Exception as e:
        traceback.print_exc()
        return ERROR_HTML.format(f"{type(e).__name__}: {e}"), ''


def do_recommend(q):
    if not q.strip():
        return '<p>Enter a product.</p>'
    try:
        source, recs = recommend(q, k=5)
        if len(recs) == 0:
            return '<p>No recommendations.</p>'
        recs['score'] = recs['similarity']
        header = (f"<p style='font-size:13px'><b>Similar to:</b> "
                  f"{str(source['product_title'])[:80]}</p>")
        return header + format_results(recs)
    except Exception as e:
        traceback.print_exc()
        return ERROR_HTML.format(f"{type(e).__name__}: {e}")


with gr.Blocks(title='DiscoverAI', theme=gr.themes.Soft(),
               css='.gradio-container{max-width:1100px!important;margin:auto}') as demo:

    gr.HTML('<div style="text-align:center;padding:16px 0 8px">'
            '<h1 style="font-size:26px;margin:0">DiscoverAI</h1>'
            '<p style="color:#718096;margin:4px 0 0;font-size:14px">'
            'Semantic Product Search &middot; Health &amp; Personal Care &middot; Deloitte &times; LUISS 2026</p>'
            '</div>')

    with gr.Tabs():
        with gr.Tab('Search'):
            with gr.Row():
                with gr.Column(scale=4):
                    qbox = gr.Textbox(label='Query', placeholder='e.g. affordable moisturizer for sensitive skin...', lines=1)
                with gr.Column(scale=1):
                    sbtn = gr.Button('Search', variant='primary')
            with gr.Row():
                nres = gr.Slider(3, 10, value=5, step=1, label='Results')
                pdd  = gr.Dropdown(['All','Budget','Low','Mid','High','Premium'], value='All', label='Price')
                rdd  = gr.Dropdown(['None','3.0','3.5','4.0','4.5'], value='3.5', label='Min rating')
                scb  = gr.Checkbox(label='Show score breakdown', value=False,
                                   info='Show similarity / quality / popularity columns')
            rout = gr.HTML()
            gr.Markdown('#### Top result detail')
            dout = gr.HTML()
            sbtn.click(do_search, [qbox,nres,pdd,rdd,scb], [rout,dout])
            qbox.submit(do_search, [qbox,nres,pdd,rdd,scb], [rout,dout])
            gr.Examples(examples=[
                ['affordable moisturizer for sensitive skin'],
                ['best electric toothbrush whitening'],
                ['natural sleep supplement without melatonin'],
                ['baby lotion fragrance free'],
                ['vitamin C 1000mg supplement'],
                ['pain relief cream for joints arthritis'],
                ['budget sunscreen SPF 50'],
                ['help me sleep naturally'],
                ['protect skin from sun at the beach'],
                ['grow longer stronger hair'],
            ], inputs=qbox)

        with gr.Tab('Similar products'):
            gr.Markdown('Enter a product name to find the most similar items in the catalogue.')
            rbox  = gr.Textbox(label='Product', placeholder='e.g. Oral-B electric toothbrush...')
            rbtn  = gr.Button('Find similar', variant='primary')
            rout2 = gr.HTML()
            rbtn.click(do_recommend, rbox, rout2)
            rbox.submit(do_recommend, rbox, rout2)
            gr.Examples(examples=[
                ['vitamin C supplement rose hips'],
                ['electric toothbrush rechargeable'],
                ['air purifier HEPA bedroom'],
                ['baby lotion Johnson'],
                ['pain relief cream arthritis'],
            ], inputs=rbox)

        with gr.Tab('System info'):
            n_summ = int(idx['summary_full'].notna().sum()) if has_summaries else 0
            n_img  = int(idx['image_url'].notna().sum())    if has_images    else 0
            gr.HTML(
                '<div style="padding:16px;font-size:13px">'
                '<h3 style="font-size:16px;font-weight:500">DiscoverAI system</h3>'
                '<table style="border-collapse:collapse;width:100%;max-width:600px">'
                f'<tr><td style="padding:5px 16px 5px 0;color:#718096">Products</td><td><b>{len(idx):,}</b></td></tr>'
                '<tr><td style="padding:5px 16px 5px 0;color:#718096">Model</td><td>all-mpnet-base-v2 &middot; 768 dim</td></tr>'
                '<tr><td style="padding:5px 16px 5px 0;color:#718096">FAISS</td><td>IndexFlatIP &middot; exact search</td></tr>'
                '<tr><td style="padding:5px 16px 5px 0;color:#718096">Re-ranking</td><td>similarity + quality + popularity</td></tr>'
                '<tr><td style="padding:5px 16px 5px 0;color:#718096">Query features</td><td>price intent &middot; negation &middot; synonyms &middot; dosage</td></tr>'
                '<tr><td style="padding:5px 16px 5px 0;color:#718096">Min rating</td><td>3.5 &#11088; (default)</td></tr>'
                f'<tr><td style="padding:5px 16px 5px 0;color:#718096">BART summaries</td><td>{"OK - " + str(n_summ) + " products" if has_summaries else "Not available"}</td></tr>'
                f'<tr><td style="padding:5px 16px 5px 0;color:#718096">Entity recognition</td><td>{"OK" if has_entities else "Not available"}</td></tr>'
                f'<tr><td style="padding:5px 16px 5px 0;color:#718096">Product images</td><td>{"OK - " + f"{n_img:,}" + " products" if has_images else "Not available"}</td></tr>'
                '</table></div>'
            )

print('Gradio layout ready.')


Gradio layout ready.


## 5 . Launch

Local-only by default (`share=False`). Set `share=True` if a public Gradio link is needed.

In [ ]:
demo.launch(share=False, inbrowser=True, server_port=7860, show_error=True)
